In [ ]:
import os
import json
from datasets import Dataset
from PIL import Image

BASE_PATH = "FATURA2/invoices_dataset_final"
ANN_DIR = os.path.join(BASE_PATH, "Annotations/layoutlm_HF_format")
IMG_DIR = os.path.join(BASE_PATH, "images")

fatura_labels = [
    "company", "date", "address", "total", "invoice_info", 
    "customer_info", "billing_info", "shipping_info", "item_description", 
    "item_unit_price", "item_qty", "item_total", "tax_info", "other"
]

id2label = {i: label for i, label in enumerate(fatura_labels)}
label2id = {label: i for i, label in enumerate(fatura_labels)}

In [ ]:
from transformers import LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

def load_fatura_paths_only():
    data = []
    json_files = [f for f in os.listdir(ANN_DIR) if f.endswith(".json")]
    for f in json_files:
        img_name = f.replace("_hugg_train.json", ".jpg")
        img_p = os.path.join(IMG_DIR, img_name)
        if os.path.exists(img_p):
            with open(os.path.join(ANN_DIR, f), 'r', encoding='utf-8') as file:
                ann = json.load(file)
            data.append({
                "image_path": img_p, # Store path, not the object
                "tokens": ann["words"],
                "bboxes": ann["bboxes"],
                "ner_tags": ann["ner_tags"]
            })
    return Dataset.from_list(data)

# Preprocess single items to save RAM
def preprocess_on_fly(examples):
    images = [Image.open(path).convert("RGB") for path in examples["image_path"]]
    encoding = processor(
        images, 
        examples["tokens"], 
        boxes=examples["bboxes"], 
        word_labels=examples["ner_tags"], 
        truncation=True, 
        padding="max_length", 
        max_length=512
    )
    return encoding

raw_dataset = load_fatura_paths_only()
# DO NOT use .map() with batched=True here if RAM is low. 
# We will let the Trainer handle the transformation or map with a small batch size.
train_dataset = raw_dataset.map(preprocess_on_fly, batched=True, batch_size=2, remove_columns=raw_dataset.column_names)
print(f"✅ Loaded {len(train_dataset)} samples safely.")

**STAGE 1**

Pretrain the LayoutLMv3 on the FATURA dataset.

In [ ]:
from transformers import LayoutLMv3ForTokenClassification, TrainingArguments, Trainer
import torch

torch.cuda.empty_cache()
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(fatura_labels),
    id2label=id2label,
    label2id=label2id
)

# Freeze the Visual Backbone (Optional but saves significant VRAM)
for param in model.layoutlmv3.patch_embed.parameters():
    param.requires_grad = False

model.to("cuda")

training_args = TrainingArguments(
    output_dir="./models/stage1_fatura",
    max_steps=1000,
    
    # --- AGGRESSIVE VRAM MANAGEMENT ---
    per_device_train_batch_size=1,   
    gradient_accumulation_steps=16,  # Increased to keep stable learning
    fp16=True,                       # Critical for RTX 3050
    
    # --- MEMORY STABILITY ---
    dataloader_num_workers=0,        # Prevents CPU forking collapse
    max_grad_norm=0.3,               # Prevents spikes that cause OOM
    
    # --- LOGGING & SAVING ---
    learning_rate=5e-5,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor.tokenizer
)

print("🚀 Starting Stage 1 Training (Manual Memory Squeeze)...")
trainer.train()
trainer.save_model("./models/fatura_backbone_v1")

**STAGE 2**

Transfer Learning on VAIPE-P dataset.

In [1]:
import torch, os
os.environ['CUDA_VISIBLE_DEVICES'] = "1"
from transformers import LayoutLMv3ForTokenClassification
torch.cuda.empty_cache()
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
prescription_labels = ["date", "diagnose", "usage", "quantity", "drugname", "other"]
id2label_p = {i: l for i, l in enumerate(prescription_labels)}
label2id_p = {l: i for i, l in enumerate(prescription_labels)}

model_s2 = LayoutLMv3ForTokenClassification.from_pretrained(
    "./models/fatura_backbone_v1",
    num_labels=6, 
    id2label=id2label_p,
    label2id=label2id_p,
    ignore_mismatched_sizes=True 
)

model_s2.to("cuda")

/data/axnguyen/Env/Anaconda/envs/vaipe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You passed `num_labels=6` which is incompatible to the `id2label` map of length `14`.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 10427.47it/s]
LayoutLMv3ForTokenClassification LOAD REPORT from: ./models/fatura_backbone_v1
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.bias            | MISSING    | 
classifier.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params wer

LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [2]:
import os
import json
from datasets import Dataset

def load_vaipe_p_dataset(base_path):
    data = []
    # Updated paths based on your folder structure
    img_dir = os.path.join(base_path, "prescription/image")
    ann_dir = os.path.join(base_path, "prescription/label")
    
    if not os.path.exists(ann_dir):
        print(f"Error: {ann_dir} not found.")
        return Dataset.from_list([])

    json_files = [f for f in os.listdir(ann_dir) if f.endswith(".json")]
    
    for f in json_files:
        img_name = f.replace(".json", ".jpg")
        img_p = os.path.join(img_dir, img_name)
        
        # Check both extensions
        if not os.path.exists(img_p):
             img_p = os.path.join(img_dir, f.replace(".json", ".png"))
        
        if os.path.exists(img_p):
            with open(os.path.join(ann_dir, f), 'r', encoding='utf-8') as file:
                ann = json.load(file)
            
            # --- FIX FOR TYPEERROR ---
            # If 'ann' is a list, the keys are likely inside the first element 
            # or the data structure is different. 
            # Based on common HF formats, if it's a list, we treat it as the item itself.
            if isinstance(ann, list):
                # If the list itself contains the word/box info, 
                # we usually need to extract them from each dictionary in that list.
                data.append({
                    "image_path": img_p,
                    "tokens": [item["text"] for item in ann],
                    "bboxes": [item["box"] for item in ann],
                    "ner_tags": [item["label"] for item in ann]
                })
            else:
                # If it's a dictionary (like your Stage 1 files)
                data.append({
                    "image_path": img_p,
                    "tokens": ann.get("words", []),
                    "bboxes": ann.get("bboxes", []),
                    "ner_tags": ann.get("ner_tags", [])
                })
    return Dataset.from_list(data)

# Re-load the real-world data
train_raw = load_vaipe_p_dataset("public_train")
test_raw = load_vaipe_p_dataset("public_test")

print(f"✅ Loaded {len(train_raw)} Training and {len(test_raw)} Test prescriptions.")

✅ Loaded 1173 Training and 172 Test prescriptions.


In [3]:
train_test_split = train_raw.train_test_split(test_size=0.2, seed=42)
temp_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)
train_dataset = train_test_split['train']
val_dataset = temp_split['train']
local_test_dataset = temp_split['test']

print(f"Training:   {len(train_dataset)} samples")
print(f"Validation: {len(val_dataset)} samples")
print(f"Local Test: {len(local_test_dataset)} samples")

Training:   938 samples
Validation: 117 samples
Local Test: 118 samples


In [4]:
from PIL import Image
from transformers import LayoutLMv3Processor, TrainingArguments, Trainer


def preprocess_stage2(examples):
    images = [Image.open(path).convert("RGB") for path in examples["image_path"]]
    
    # 1. Label Guard (0-5)
    cleaned_labels = []
    for label_list in examples["ner_tags"]:
        cleaned_labels.append([min(label2id_p.get(l, 5) if isinstance(l, str) else l, 5) for l in label_list])
    
    # 2. Spatial Guard (0-1000)
    cleaned_boxes = []
    for box_list in examples["bboxes"]:
        safe_boxes = []
        for box in box_list:
            # Strictly clamp every coordinate to [0, 1000]
            safe_box = [max(0, min(1000, int(coord))) for coord in box]
            safe_boxes.append(safe_box)
        cleaned_boxes.append(safe_boxes)
    
    return processor(
        images, 
        examples["tokens"], 
        boxes=cleaned_boxes, 
        word_labels=cleaned_labels, 
        truncation=True, 
        padding="max_length", 
        max_length=512  # <-- Increased from 224 to 512 for max accuracy
    )

# Re-map the datasets with this new "Safe" logic
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
train_ds = train_dataset.map(preprocess_stage2, batched=True, batch_size=2)
val_ds = val_dataset.map(preprocess_stage2, batched=True, batch_size=2)
local_test_ds = local_test_dataset.map(preprocess_stage2, batched=True, batch_size=2)


Map: 100%|██████████| 118/118 [00:09<00:00, 12.45 examples/s]


In [5]:
from transformers import LayoutLMv3Processor, TrainingArguments, Trainer

training_args_final = TrainingArguments(
    output_dir="./models/vaipe_811_split",
    max_steps=2000,
    
    # --- UNLEASHING 24GB VRAM ---
    per_device_train_batch_size=12,  # Increased from 1 to drastically speed up epochs
    gradient_accumulation_steps=1,   # Reduced from 16 since real batch size is huge
    per_device_eval_batch_size=12,   # Fast evaluation
    
    learning_rate=2e-5,
    fp16=True,                       # Keep fp16 for Tensor Core acceleration on A5500
    
    # --- ALIGNMENT FIX ---
    eval_strategy="steps",
    eval_steps=100,                  # Lowered to 100 since steps will compute 12x faster
    save_strategy="steps",          
    save_steps=100,                  
    
    save_total_limit=2,
    load_best_model_at_end=True,     
    metric_for_best_model="eval_loss",
    
    # --- CPU MULTITHREADING ---
    dataloader_num_workers=0         # Use robust CPU loading to feed the A5500 quickly
)

trainer_final = Trainer(
    model=model_s2,
    args=training_args_final,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor.tokenizer
)


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [6]:
# Check the actual DataLoader to see what is about to hit the GPU
train_dataloader = trainer_final.get_train_dataloader()
batch = next(iter(train_dataloader))

# Labels check
labels = batch["labels"]
max_label = labels.max().item()
min_label = labels.min().item()

print(f"📊 Tensor Inspection:")
print(f"   - Max Label ID: {max_label}")
print(f"   - Min Label ID: {min_label} (Expected -100 for padding)")
print(f"   - Model Config Labels: {model_s2.config.num_labels}")

if max_label >= model_s2.config.num_labels:
    print(f"CRITICAL: A label ID ({max_label}) exceeds the model's head size (6)!")
    # Find where the bad label is
    bad_indices = (labels == max_label).nonzero(as_tuple=True)
    print(f"   Check mapping for these tokens immediately.")
else:
    print("DATA ALIGNMENT OK: Labels are within the 0-5 range.")

📊 Tensor Inspection:
   - Max Label ID: 5
   - Min Label ID: -100 (Expected -100 for padding)
   - Model Config Labels: 6
DATA ALIGNMENT OK: Labels are within the 0-5 range.


In [ ]:
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = "1" 
trainer_final.train()
trainer_final.save_model("./models/vaipe_final_best")
processor.save_pretrained("./models/vaipe_final_best")

Step,Training Loss,Validation Loss
100,No log,0.014364
200,No log,0.012173
300,No log,0.010918
400,No log,0.009412
500,0.045698,0.011001


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.47s/it]


In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers.utils.notebook import NotebookProgressCallback

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index -100 and map to label names
    true_predictions = [
        [id2label_p[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label_p[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": np.mean([p == l for p_list, l_list in zip(true_predictions, true_labels) 
                             for p, l in zip(p_list, l_list)])
    }

trainer_final.compute_metrics = compute_metrics
trainer_final.pop_callback(NotebookProgressCallback)
test_results = trainer_final.evaluate(local_test_ds)

print(f"F1:  {test_results['eval_f1']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")

In [ ]:
from transformers import LayoutLMv3Processor
from PIL import Image

# 1. Load the Processor with OCR enabled
print("Loading processor...")
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=True)

# 2. Open an image from your test dataset
image_path = "public_test/prescription/image/VAIPE_P_TEST_0.png"
image = Image.open(image_path).convert("RGB")

print(f"Running OCR on {image_path}...")
try:
    # 3. Process the image (Notice we ONLY pass the image, no text/boxes)
    encoding = processor(image, return_tensors="pt")
    
    # 4. Decode the tokens back to words to see what it found!
    input_ids = encoding["input_ids"].squeeze().tolist()
    extracted_words = processor.tokenizer.convert_ids_to_tokens(input_ids)
    
    # Clean up special tokens for readability
    clean_words = [word for word in extracted_words if word not in ["<s>", "</s>", "<pad>"]]
    
    print("\n✅ SUCCESS! Here are the words the model's OCR extracted:")
    print("-" * 50)
    print(" ".join(clean_words).replace("Ġ", " "))  # Ġ represents spaces in this tokenizer
    print("-" * 50)

except Exception as e:
    print("\n❌ OCR FAILED. Error details:")
    print(e)


In [ ]:
import os
import json
import torch
from PIL import Image
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification

# 1. Provide Context & Paths
test_img_dir = "public_test/prescription/image"
output_label_dir = "public_test/prescription/label"

# We must use checkpoint-1400 or checkpoint-1200 because the trainer deleted 2000 to save space
checkpoint_path = "./models/vaipe_811_split/checkpoint-1400"

os.makedirs(output_label_dir, exist_ok=True)

# Re-define our local labels since kernel restarted
prescription_labels = ["date", "diagnose", "usage", "quantity", "drugname", "other"]
id2label_p = {i: l for i, l in enumerate(prescription_labels)}
label2id_p = {l: i for i, l in enumerate(prescription_labels)}

# 2. Load the Processor & Model from disk
print("Loading Processor with OCR...")
processor_infer = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=True, ocr_lang="vie")

print(f"Loading Model from {checkpoint_path}...")
model_infer = LayoutLMv3ForTokenClassification.from_pretrained(checkpoint_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_infer.to(device)
model_infer.eval() 

# 3. Define helper to un-normalize bounding boxes
def unnormalize_box(bbox, width, height):
    return [
        int(bbox[0] * width / 1000.0), int(bbox[1] * height / 1000.0),
        int(bbox[2] * width / 1000.0), int(bbox[3] * height / 1000.0),
    ]

# 4. Fetch images
image_files = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(image_files)} images, starting inference...\n")

for img_file in image_files:
    img_path = os.path.join(test_img_dir, img_file)
    image = Image.open(img_path).convert("RGB")
    width, height = image.size
    
    encoding = processor_infer(image, return_tensors="pt", truncation=True, padding="max_length", max_length=512)
    
    for k, v in encoding.items():
        if isinstance(v, torch.Tensor):
            encoding[k] = v.to(device)
            
    with torch.no_grad():
        outputs = model_infer(**encoding)
        
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    token_boxes = encoding.bbox.squeeze().tolist()
    input_ids = encoding.input_ids.squeeze().tolist()
    word_ids = encoding.word_ids(batch_index=0)
    
    json_data = []
    item_id = 1
    previous_word_idx = None
    
    current_word = ""
    current_box = []
    current_label = None
    
    for i, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
            
        token = processor_infer.tokenizer.decode([input_ids[i]])
        label_id = predictions[i]
        label_name = id2label_p.get(label_id, "other") 
        box = token_boxes[i]
        
        if word_idx != previous_word_idx:
            if previous_word_idx is not None:
                json_data.append({
                    "id": item_id,
                    "text": current_word.strip(),
                    "label": current_label,
                    "box": unnormalize_box(current_box, width, height)
                })
                item_id += 1
                
            current_word = token.replace("##", "")
            current_box = box 
            current_label = label_name 
        else:
            current_word += token.replace("##", "")
            
        previous_word_idx = word_idx
        
    if previous_word_idx is not None:
        json_data.append({
            "id": item_id,
            "text": current_word.strip(),
            "label": current_label,
            "box": unnormalize_box(current_box, width, height)
        })

    # ========================================================
    # NEW CODE: MERGE ADJACENT WORDS WITH THE SAME LABEL
    # ========================================================
    merged_data = []
    
    for item in json_data:
        if not merged_data:
            merged_data.append(item)
            continue
            
        last = merged_data[-1]
        
        # If this word has the same label as the last word (and is not an irrelevant "other")
        # Then squish them together into a GROUP of words (a phrase)!
        if item["label"] == last["label"] and item["label"] != "other":
            last["text"] += " " + item["text"]
            
            # Stretch the bounding box so it wraps around BOTH words
            last["box"][0] = min(last["box"][0], item["box"][0]) # min x
            last["box"][1] = min(last["box"][1], item["box"][1]) # min y
            last["box"][2] = max(last["box"][2], item["box"][2]) # max x
            last["box"][3] = max(last["box"][3], item["box"][3]) # max y
        else:
            merged_data.append(item)
            
    # Give the finalized, grouped phrases correct IDs mapping 1 to N
    new_id = 1
    for item in merged_data:
        item["id"] = new_id
        new_id += 1
    # ========================================================

    json_name = os.path.splitext(img_file)[0] + ".json"
    json_path = os.path.join(output_label_dir, json_name)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=4) 
        
print(f"✅ Extracted NER labels for test images directly into '{output_label_dir}' directory.")


In [ ]:
import os
import json
import csv

label_dir = "public_test/prescription/label"
output_csv = "public_test/prescription/predictions_summary.csv"

# Open the CSV file for writing
with open(output_csv, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    # Write the column headers
    writer.writerow(['file_name', 'item_id', 'text_token', 'ner_label', 'xmin', 'ymin', 'xmax', 'ymax'])
    
    # Iterate through all JSON files
    for filename in os.listdir(label_dir):
        # We skip hidden Mac files (like ._VAIPE_P_TEST_1.json)
        if filename.endswith(".json") and not filename.startswith("._"):
            json_path = os.path.join(label_dir, filename)
            
            with open(json_path, 'r', encoding='utf-8') as jf:
                try:
                    data = json.load(jf)
                    # Write each detected word as a row in the CSV
                    for item in data:
                        box = item.get("box", [0, 0, 0, 0])
                        writer.writerow([
                            filename,
                            item.get("id", ""),
                            item.get("text", ""),
                            item.get("label", ""),
                            box[0], box[1], box[2], box[3]
                        ])
                except json.JSONDecodeError:
                    print(f"Skipping empty or malformed file: {filename}")

print(f"✅ Successfully converted all JSON files into a single CSV at: {output_csv}")
